In [37]:
from ortools.sat.python import cp_model

In [38]:
flights = [
    {"id": "AV101", "arrival": 60,  "departure": 120},
    {"id": "LA202", "arrival": 90,  "departure": 180},
    {"id": "IB303", "arrival": 130, "departure": 210},
    {"id": "AA404", "arrival": 200, "departure": 260},
    {"id": "KL505", "arrival": 240, "departure": 320},
    {"id": "CM606", "arrival": 300, "departure": 380},
    {"id": "UX707", "arrival": 330, "departure": 410},
    {"id": "AF808", "arrival": 360, "departure": 450},
    {"id": "LH909", "arrival": 400, "departure": 500},
    {"id": "DL010", "arrival": 470, "departure": 560},
    {"id": "AM111", "arrival": 520, "departure": 610},
    {"id": "TP212", "arrival": 580, "departure": 660},
    {"id": "EK313", "arrival": 620, "departure": 710},
    {"id": "QR414", "arrival": 700, "departure": 780},
    {"id": "BA515", "arrival": 750, "departure": 840},
]

gates = [
    "G1", "G2", "G3", "G4", "G5"
]

In [39]:
model = cp_model.CpModel()

x = {}

for f in range(len(flights)):
    for g in range(len(gates)):
        x[(f, g)] = model.NewBoolVar(f"x_f{f}_g{g}")

# cada vuelo debe asignarse a exactamente una puerta
for f in range(len(flights)):
    model.Add(sum(x[(f, g)] for g in range(len(gates))) == 1)

In [40]:
# vuelos en el mismo horario no pueden compartir la misma puerta
for i in range(len(flights)):
    for j in range(i + 1, len(flights)):

        overlap = not (
            flights[i]["departure"] <= flights[j]["arrival"]
            or
            flights[j]["departure"] <= flights[i]["arrival"]
        )

        if overlap:
            for g in range(len(gates)):
                model.Add(x[(i, g)] + x[(j, g)] <= 1)

In [41]:
gate_used = []

for g in range(len(gates)):
    used = model.NewBoolVar(f"gate_used_{g}")
    gate_used.append(used)

    for f in range(len(flights)):
        model.AddImplication(x[(f, g)], used)

model.Minimize(sum(gate_used))

In [42]:
solver = cp_model.CpSolver()
status = solver.Solve(model)

In [43]:
print("\n" + "=" * 45)
print("      SISTEMA DE ASIGNACION DE PUERTAS")
print("=" * 45)

if status in [cp_model.OPTIMAL, cp_model.FEASIBLE]:

    print("\nAsignacion encontrada:\n")

    for f in range(len(flights)):

        assigned_gate = None

        for g in range(len(gates)):
            if solver.Value(x[(f, g)]) == 1:
                assigned_gate = gates[g]

        arrival = flights[f]["arrival"]
        departure = flights[f]["departure"]

        print(
            f"[{flights[f]['id']}] "
            f"Llega: {arrival:03d}  "
            f"Sale: {departure:03d}  "
            f"-> Puerta: {assigned_gate}"
        )

    print("\n" + "-" * 45)
    print("Puertas utilizadas:")

    for g in range(len(gates)):
        if solver.Value(gate_used[g]):
            print(f"  - {gates[g]}")

    print("-" * 45)

else:
    print("No se encontro solucion valida.")


      SISTEMA DE ASIGNACION DE PUERTAS

Asignacion encontrada:

[AV101] Llega: 060  Sale: 120  -> Puerta: G5
[LA202] Llega: 090  Sale: 180  -> Puerta: G1
[IB303] Llega: 130  Sale: 210  -> Puerta: G2
[AA404] Llega: 200  Sale: 260  -> Puerta: G1
[KL505] Llega: 240  Sale: 320  -> Puerta: G2
[CM606] Llega: 300  Sale: 380  -> Puerta: G1
[UX707] Llega: 330  Sale: 410  -> Puerta: G2
[AF808] Llega: 360  Sale: 450  -> Puerta: G5
[LH909] Llega: 400  Sale: 500  -> Puerta: G1
[DL010] Llega: 470  Sale: 560  -> Puerta: G2
[AM111] Llega: 520  Sale: 610  -> Puerta: G1
[TP212] Llega: 580  Sale: 660  -> Puerta: G2
[EK313] Llega: 620  Sale: 710  -> Puerta: G1
[QR414] Llega: 700  Sale: 780  -> Puerta: G2
[BA515] Llega: 750  Sale: 840  -> Puerta: G1

---------------------------------------------
Puertas utilizadas:
  - G1
  - G2
  - G5
---------------------------------------------
